# 11k — Net-zero core-satellite allocation (`net_zero16-18`)

Three closed-form (no optimizer, no external data) scripts on a
**core-satellite** decarbonization strategy: a small "satellite" sleeve
invested more aggressively (steeper equity/bond mix skew) sits alongside
a larger, more conventional "core" sleeve, and the overall portfolio's
implied bond/equity split and tracking-error volatility are derived
algebraically from the two sleeves' individual allocations and weights.

In [1]:
import numpy as np
import pandas as pd


def corr_from_lower_triangle(values, n):
    # Build a symmetric n x n correlation matrix from its lower-triangular
    # entries listed row by row, e.g. [1, r21, 1, r31, r32, 1, ...]
    # (MATLAB/GAUSS `xpnd` convention used throughout the .m source) --
    # reused as-is from 03a/03b/11a.
    corr = np.eye(n)
    it = iter(values)
    for i in range(n):
        for j in range(i + 1):
            v = next(it)
            corr[i, j] = corr[j, i] = v
    return corr

## 1. Core-sleeve bond/equity split implied by a satellite overlay (`net_zero16`)

Given an overall target bond weight and a (more bond-heavy) satellite
sleeve's own bond weight, backs out what the core sleeve's bond/equity
split must be so the blended (core + satellite) portfolio still hits the
overall target: $\alpha_{\text{core bond}} = \frac{\alpha_{\text{bond}}
- \alpha_{\text{satellite}}\,\alpha_{\text{satellite bond}}}
{1-\alpha_{\text{satellite}}}$.

In [2]:
alpha_bond = 0.40
alpha_satellite_bond = 0.70
alpha_satellite_equity = 1 - alpha_satellite_bond
alpha_satellite = 0.10
alpha_core = 1 - alpha_satellite
alpha_core_bond = (alpha_bond - alpha_satellite * alpha_satellite_bond) / (1 - alpha_satellite)
alpha_core_equity = 1 - alpha_core_bond

print("Core equity / Core bond / Satellite equity / Satellite bond (%):",
      np.round(100 * np.array([alpha_core_equity, alpha_core_bond, alpha_satellite_equity, alpha_satellite_bond]), 2))
blended_equity = alpha_core * alpha_core_equity + alpha_satellite * alpha_satellite_equity
print(f"Blended equity weight check (should reconstruct 1 - alpha_bond = {100*(1-alpha_bond):.2f}%): {100*blended_equity:.2f}%")

alpha_bond_grid = np.array([0.40, 0.40, 0.40, 0.50, 0.50, 0.50, 0.80, 0.80, 0.80])
alpha_satellite_bond_grid = np.array([0.70, 0.80, 0.90, 0.70, 0.80, 0.90, 0.70, 0.80, 0.90])
alpha_satellite_grid = np.array([0.00, 0.01, 0.05, 0.10, 0.15, 0.20, 0.25])
alpha_core_bond_grid = (alpha_bond_grid[None, :] - alpha_satellite_grid[:, None] * alpha_satellite_bond_grid[None, :]) / \
                       (1 - alpha_satellite_grid[:, None])

table16 = pd.DataFrame(100 * alpha_core_bond_grid,
                        index=[f"satellite={100*a:.0f}%" for a in alpha_satellite_grid],
                        columns=[f"bond={100*b:.0f}%/sat_bond={100*sb:.0f}%"
                                 for b, sb in zip(alpha_bond_grid, alpha_satellite_bond_grid)])
display(table16.round(1))

Core equity / Core bond / Satellite equity / Satellite bond (%): [63.33 36.67 30.   70.  ]
Blended equity weight check (should reconstruct 1 - alpha_bond = 60.00%): 60.00%


,bond=40%/sat_bond=70%,bond=40%/sat_bond=80%,bond=40%/sat_bond=90%,bond=50%/sat_bond=70%,bond=50%/sat_bond=80%,bond=50%/sat_bond=90%,bond=80%/sat_bond=70%,bond=80%/sat_bond=80%,bond=80%/sat_bond=90%
satellite=0%,40.0,40.0,40.0,50.0,50.0,50.0,80.0,80.0,80.0
satellite=1%,39.7,39.6,39.5,49.8,49.7,49.6,80.1,80.0,79.9
satellite=5%,38.4,37.9,37.4,48.9,48.4,47.9,80.5,80.0,79.5
satellite=10%,36.7,35.6,34.4,47.8,46.7,45.6,81.1,80.0,78.9
satellite=15%,34.7,32.9,31.2,46.5,44.7,42.9,81.8,80.0,78.2
satellite=20%,32.5,30.0,27.5,45.0,42.5,40.0,82.5,80.0,77.5
satellite=25%,30.0,26.7,23.3,43.3,40.0,36.7,83.3,80.0,76.7


## 2. Blended tracking-error volatility from core+satellite risk parameters (`net_zero17`)

Treats core-equity, core-bond, satellite-equity, and satellite-bond as 4
correlated risk factors with their own volatilities, blends them by the
core-satellite weights from Section 1, and evaluates the resulting
portfolio's tracking-error volatility $\sigma = \sqrt{\alpha'\Sigma\alpha}$.

In [3]:
alpha_satellite = 0.10
alpha_equity = 0.60
alpha_bond = 0.40
alpha_satellite_bond = 0.70

alpha_satellite_equity = 1 - alpha_satellite_bond
alpha_core = 1 - alpha_satellite
alpha_core_bond = (alpha_bond - alpha_satellite * alpha_satellite_bond) / (1 - alpha_satellite)
alpha_core_equity = 1 - alpha_core_bond

sigma_core_equity, sigma_core_bond = 0.02, 0.0025
sigma_satellite_equity, sigma_satellite_bond = 0.20, 0.03
sigma = np.array([sigma_core_equity, sigma_core_bond, sigma_satellite_equity, sigma_satellite_bond])
rho = corr_from_lower_triangle([1.00,
                                 0.00, 1.00,
                                 0.80, 0.00, 1.00,
                                 0.00, 0.50, 0.00, 1.00], 4)
Sigma17 = rho * np.outer(sigma, sigma)
print("Sigma (x 1e4):")
display(pd.DataFrame(1e4 * Sigma17, index=["core_eq", "core_bd", "sat_eq", "sat_bd"],
                      columns=["core_eq", "core_bd", "sat_eq", "sat_bd"]).round(4))

alpha17 = np.array([(1 - alpha_satellite) * alpha_core_equity, (1 - alpha_satellite) * alpha_core_bond,
                     alpha_satellite * alpha_satellite_equity, alpha_satellite * alpha_satellite_bond])
print("alpha (%):", np.round(1e2 * alpha17, 4))
sigma_te17 = np.sqrt(alpha17 @ Sigma17 @ alpha17)
print(f"sigma_te = {sigma_te17:.6f}  ({1e4*sigma_te17:.2f} bps)")

Sigma (x 1e4):


,core_eq,core_bd,sat_eq,sat_bd
core_eq,4.0,0.0000,32.0,0.000
core_bd,0.0,0.0625,0.0,0.375
sat_eq,32.0,0.0000,400.0,0.000
sat_bd,0.0,0.3750,0.0,9.000


alpha (%): [57. 33.  3.  7.]
sigma_te = 0.016799  (167.99 bps)


## 3. Tracking-error volatility across a satellite-weight / equity-split grid (`net_zero18`)

Same 4-factor blended tracking-error calculation as Section 2, but swept
across a grid of satellite weights (10/20/30%) and equity/bond splits
(0-100%), under two correlation scenarios: `Sigma1` uses a fully
uncorrelated ($\rho=I$) 4-factor covariance -- no core-satellite
correlation at all, not even within the same asset class -- while
`Sigma2` is *intended* (per its own `rho`/`xpnd(rho)` lines) to add
$\rho=0.8$ equity-equity correlation between the core and satellite
sleeves, but see the bug note below.

**Faithfully reproduced quirk**: the source's `Sigma2` line is
`Sigma2 = sigma .* sigma'` -- it does **not** multiply by the
correlation matrix `rho` at all, so the `rho = [...]; rho = xpnd(rho);`
lines immediately above it are dead code and `Sigma2` is actually an
**all-pairs-perfectly-correlated** ($\rho\equiv1$) covariance matrix,
not the intended $\rho=0.8$-equity-cross-correlation scenario its
surrounding code suggests. This looks like a copy-paste bug in the
original script (very likely meant to read
`Sigma2 = rho .* sigma .* sigma'`, mirroring `Sigma1`'s line just
above it) -- reproduced literally below rather than silently corrected,
since the point of this notebook series is a faithful port.

In [4]:
sigma = np.array([0.02, 0.0025, 0.20, 0.03])
rho1 = corr_from_lower_triangle([1.00,
                                  0.00, 1.00,
                                  0.00, 0.00, 1.00,
                                  0.00, 0.00, 0.00, 1.00], 4)
Sigma1_18 = rho1 * np.outer(sigma, sigma)

rho2 = corr_from_lower_triangle([1.00,
                                  0.00, 1.00,
                                  0.80, 0.00, 1.00,
                                  0.00, 0.80, 0.00, 1.00], 4)
# Faithful to the source: Sigma2 does NOT use rho2 (dead code above), see markdown note.
Sigma2_18 = np.outer(sigma, sigma)

alpha_satellite_vectors = np.array([0.10, 0.20, 0.30])
alpha_equity_vectors = np.array([0.00, 0.20, 0.50, 0.60, 0.80, 1.00])

result1 = np.zeros((3, 6))
result2 = np.zeros((3, 6))
for i, a_sat in enumerate(alpha_satellite_vectors):
    for j, a_eq in enumerate(alpha_equity_vectors):
        alpha_core_equity = a_eq
        alpha_satellite_equity = a_eq
        alpha_core_bond = 1 - a_eq
        alpha_satellite_bond = 1 - a_eq
        alpha18 = np.array([(1 - a_sat) * alpha_core_equity, (1 - a_sat) * alpha_core_bond,
                             a_sat * alpha_satellite_equity, a_sat * alpha_satellite_bond])
        result1[i, j] = np.sqrt(alpha18 @ Sigma1_18 @ alpha18)
        result2[i, j] = np.sqrt(alpha18 @ Sigma2_18 @ alpha18)

cols = [f"equity={100*a:.0f}%" for a in alpha_equity_vectors]
idx = [f"satellite={100*a:.0f}%" for a in alpha_satellite_vectors]
print("Sigma1 scenario (fully uncorrelated, rho=I), sigma_te (%):")
display(pd.DataFrame(100 * result1, index=idx, columns=cols).round(2))
print("Sigma2 scenario (all pairs rho=1, per the source's dead-code quirk above), sigma_te (%):")
display(pd.DataFrame(100 * result2, index=idx, columns=cols).round(2))

Sigma1 scenario (fully uncorrelated, rho=I), sigma_te (%):


,equity=0%,equity=20%,equity=50%,equity=60%,equity=80%,equity=100%
satellite=10%,0.38,0.62,1.36,1.62,2.15,2.69
satellite=20%,0.63,1.00,2.18,2.60,3.45,4.31
satellite=30%,0.92,1.43,3.11,3.71,4.93,6.16


Sigma2 scenario (all pairs rho=1, per the source's dead-code quirk above), sigma_te (%):


,equity=0%,equity=20%,equity=50%,equity=60%,equity=80%,equity=100%
satellite=10%,0.52,1.18,2.16,2.49,3.15,3.8
satellite=20%,0.80,1.76,3.20,3.68,4.64,5.6
satellite=30%,1.08,2.34,4.24,4.87,6.14,7.4
